# SmartFuel Forecast — Machine Learning Model v0

## 1. Project Objective

The objective is to predict whether fuel prices at Spanish petrol stations will increase, decrease, or remain stable.

The model uses historical fuel-price observations collected from the Spanish Ministry's API and stored in AWS S3.

We will:
1. Load and explore the fuel-price data.
2. Prepare historical data for training.
3. Establish a baseline model (no price change).
4. Train a Machine Learning classification model.
5. Evaluate its performance.
6. Save the trained model for deployment on AWS.


In [1]:
import pandas as pd
from pathlib import Path

# Locate the existing dataset
filename = "fuel_prices_2026-11-04_1325.csv"

possible_paths = [
    Path("data/processed") / filename,
    Path("../data/processed") / filename
]

data_path = next(
    (p for p in possible_paths if p.exists()),
    None
)

if data_path is None:
    raise FileNotFoundError("Fuel-price dataset not found.")

# Load the data
df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst rows:")
display(df.head())

Dataset shape: (11465, 10)

Columns:
['station_id', 'station_name', 'address', 'municipality', 'province', 'latitude', 'longitude', 'gasoline_95_price', 'diesel_price', 'snapshot_date']

First rows:


,station_id,station_name,address,municipality,province,latitude,longitude,gasoline_95_price,diesel_price,snapshot_date
0,4375,Nº 10.935,"AVENIDA CASTILLA LA MANCHA, 26",Abengibre,ALBACETE,39.211417,-1.539167,1.699,1.839,2026-11-04 13:25:51
1,5122,REPSOL,"CR CM-332, 46,4",Alatoz,ALBACETE,39.100389,-1.346083,1.939,1.999,2026-11-04 13:25:51
2,12054,BP ROMICA,CALLE PRINCIPE DE ASTURIAS (POLÍGONO DE ROMICA...,Albacete,ALBACETE,39.054694,-1.832000,1.759,1.949,2026-11-04 13:25:51
3,13933,PLENERGY,"CALLE FEDERICO GARCIA LORCA, 1",Albacete,ALBACETE,39.000861,-1.849833,1.615,1.729,2026-11-04 13:25:51
4,10765,CARREFOUR,"AVENIDA 1º DE MAYO, S/N",Albacete,ALBACETE,38.985667,-1.868500,1.739,1.899,2026-11-04 13:25:51



## 2. Data Quality and Exploration

Before training the model, we examine the dataset to identify missing values, duplicate records, and the available historical observations.

This step is important because predicting price movements requires multiple observations of the same petrol station over time.


In [2]:

# Convert the snapshot date to datetime
df["snapshot_date"] = pd.to_datetime(
    df["snapshot_date"], errors="coerce"
)

# Convert fuel prices to numeric values
for col in ["gasoline_95_price", "diesel_price"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Explore the dataset
print("Total records:", len(df))
print("Unique stations:", df["station_id"].nunique())
print("Unique snapshot timestamps:", df["snapshot_date"].nunique())

print("\nDate range:")
print("First:", df["snapshot_date"].min())
print("Last:", df["snapshot_date"].max())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate records:", df.duplicated().sum())

print("\nFuel price statistics:")
display(df[["gasoline_95_price", "diesel_price"]].describe())


Total records: 11465
Unique stations: 11465
Unique snapshot timestamps: 1

Date range:
First: 2026-11-04 13:25:51
Last: 2026-11-04 13:25:51

Missing values:
station_id             0
station_name           0
address                0
municipality           0
province               0
latitude               0
longitude              0
gasoline_95_price    566
diesel_price         219
snapshot_date          0
dtype: int64

Duplicate records: 0

Fuel price statistics:


,gasoline_95_price,diesel_price
count,10899.000000,11246.000000
mean,1.708847,1.891002
std,0.097675,0.104352
min,1.345000,1.381000
25%,1.649000,1.799000
50%,1.729000,1.925000
75%,1.779000,1.979000
max,2.299000,2.299000



## 3. Data Preparation

The original dataset contains one row per petrol station, with separate columns for gasoline and diesel prices.

We transform it into a long format, where each row represents one station, one fuel type and one observation timestamp.

This structure is compatible with the SmartFuel data contract and will allow us to compare prices over time.

Missing fuel prices are excluded, as they do not represent valid price observations.


In [3]:

# Define the fuel types according to SmartFuel contracts
fuel_mapping = {
    "gasoline_95_price": "gasolina_95_e5",
    "diesel_price": "gasoleo_a"
}

# Transform from wide format to long format
fuel_df = df.melt(
    id_vars=[
        "station_id",
        "station_name",
        "municipality",
        "province",
        "snapshot_date"
    ],
    value_vars=list(fuel_mapping.keys()),
    var_name="source_fuel",
    value_name="price_eur"
)

# Standardize fuel names
fuel_df["fuel_type"] = fuel_df["source_fuel"].map(fuel_mapping)

# Remove missing or invalid prices
fuel_df = fuel_df.dropna(
    subset=["price_eur", "snapshot_date"]
)
fuel_df = fuel_df[fuel_df["price_eur"] > 0].copy()

# Match SmartFuel's naming conventions
fuel_df = fuel_df.rename(
    columns={"snapshot_date": "ts"}
)
fuel_df = fuel_df.drop(columns=["source_fuel"])

# Make station identifiers strings
fuel_df["station_id"] = fuel_df["station_id"].astype(str)

print("Valid fuel-price observations:", len(fuel_df))

print("\nObservations by fuel type:")
print(fuel_df["fuel_type"].value_counts())

print("\nAvailable timestamps:", fuel_df["ts"].nunique())

display(fuel_df.head())


Valid fuel-price observations: 22145

Observations by fuel type:
fuel_type
gasoleo_a         11246
gasolina_95_e5    10899
Name: count, dtype: int64

Available timestamps: 1


,station_id,station_name,municipality,province,ts,price_eur,fuel_type
0,4375,Nº 10.935,Abengibre,ALBACETE,2026-11-04 13:25:51,1.699,gasolina_95_e5
1,5122,REPSOL,Alatoz,ALBACETE,2026-11-04 13:25:51,1.939,gasolina_95_e5
2,12054,BP ROMICA,Albacete,ALBACETE,2026-11-04 13:25:51,1.759,gasolina_95_e5
3,13933,PLENERGY,Albacete,ALBACETE,2026-11-04 13:25:51,1.615,gasolina_95_e5
4,10765,CARREFOUR,Albacete,ALBACETE,2026-11-04 13:25:51,1.739,gasolina_95_e5



## 4. Baseline Model — No Price Change

Our baseline assumes that the next observed fuel price will be equal to the current price.

- Predicted next price = Current price
- Predicted direction = Flat

This baseline will be used to evaluate whether our Machine Learning model provides better predictions.

The baseline can be generated with one snapshot, but its accuracy cannot be evaluated until future observations are available.


In [4]:

# Create baseline predictions
baseline_df = fuel_df[
    ["station_id", "fuel_type", "ts", "price_eur"]
].copy()

# Assume the next fuel price will remain unchanged
baseline_df["baseline_next_price"] = baseline_df["price_eur"]
baseline_df["baseline_direction"] = "flat"

print("Baseline predictions generated:", len(baseline_df))

display(baseline_df.head())


Baseline predictions generated: 22145


,station_id,fuel_type,ts,price_eur,baseline_next_price,baseline_direction
0,4375,gasolina_95_e5,2026-11-04 13:25:51,1.699,1.699,flat
1,5122,gasolina_95_e5,2026-11-04 13:25:51,1.939,1.939,flat
2,12054,gasolina_95_e5,2026-11-04 13:25:51,1.759,1.759,flat
3,13933,gasolina_95_e5,2026-11-04 13:25:51,1.615,1.615,flat
4,10765,gasolina_95_e5,2026-11-04 13:25:51,1.739,1.739,flat



## 5. Historical Data Preparation

To train the ML model, we need multiple price observations for each petrol station and fuel type.

We will:
1. Sort observations chronologically.
2. Identify the next observed price for each station and fuel type.
3. Calculate the price variation.
4. Classify each variation as "up", "down", or "flat".
5. Prepare the target variable for supervised learning.

Currently, only one snapshot is available. We will calculate the targets once additional historical snapshots have been collected.


In [5]:

import numpy as np

# Prepare observations for historical comparison
history_df = fuel_df[
    ["station_id", "fuel_type", "ts", "price_eur"]
].copy()

# Sort by petrol station, fuel type and timestamp
history_df = history_df.sort_values(
    ["station_id", "fuel_type", "ts"]
)

# Identify the next observed price for each station and fuel
group = history_df.groupby(["station_id", "fuel_type"])

history_df["next_price_eur"] = group["price_eur"].shift(-1)
history_df["next_ts"] = group["ts"].shift(-1)

# Keep observations for which a future price is available
training_df = history_df.dropna(
    subset=["next_price_eur"]
).copy()

# Calculate observed price changes
training_df["price_change_eur"] = (
    training_df["next_price_eur"] - training_df["price_eur"]
).round(3)

# Classify the change
training_df["direction"] = np.select(
    [
        training_df["price_change_eur"] > 0,
        training_df["price_change_eur"] < 0
    ],
    ["up", "down"],
    default="flat"
)

print("Historical observations:", len(history_df))
print("Training examples available:", len(training_df))

if training_df.empty:
    print("Not enough historical snapshots to train the model yet.")
else:
    display(training_df.head())
    print(training_df["direction"].value_counts())


Historical observations: 22145
Training examples available: 0
Not enough historical snapshots to train the model yet.



## 6. Load Historical Fuel Price Data

Load multiple historical fuel-price snapshots collected from the Spanish Ministry's API.

These observations will allow us to analyze price changes over time and prepare the dataset for training our ML model.


In [1]:

import json
from pathlib import Path
import pandas as pd

# Folder containing Marc's historical data
history_dir = Path.home() / "Desktop" / "smartfuel-raw"

# Find all JSON files, including those inside subfolders
json_files = sorted(history_dir.rglob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"No JSON files found in {history_dir}. "
        "Check that the ZIP is extracted."
    )

# Read information from each snapshot
snapshots = []

for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    snapshots.append({
        "file": file.name,
        "timestamp": pd.to_datetime(data["Fecha"], dayfirst=True),
        "stations": len(data["ListaEESSPrecio"])
    })

# Summary of available historical data
history_summary = pd.DataFrame(snapshots)
history_summary = history_summary.sort_values("timestamp")

print("Historical snapshots:", len(history_summary))
print("Unique timestamps:", history_summary["timestamp"].nunique())
print("First snapshot:", history_summary["timestamp"].min())
print("Last snapshot:", history_summary["timestamp"].max())

display(history_summary)


Historical snapshots: 23
Unique timestamps: 23
First snapshot: 2026-11-12 11:27:41
Last snapshot: 2026-11-17 13:00:08


,file,timestamp,stations
0,1127.json,2026-11-12 11:27:41,11485
1,1138.json,2026-11-12 11:38:44,11485
2,1139.json,2026-11-12 11:39:06,11485
3,1300.json,2026-11-12 13:00:08,11488
4,1900.json,2026-11-12 19:00:08,11505
5,0100.json,2026-11-13 01:00:08,11500
6,0700.json,2026-11-13 07:00:08,11500
7,1300.json,2026-11-13 13:00:08,11500
8,1900.json,2026-11-13 19:00:08,11501
9,0100.json,2026-11-14 01:00:08,11504



## 7. Build the Historical Price Dataset

Convert the 23 historical JSON snapshots into a single dataset containing fuel prices observed at different timestamps.

Each row represents one petrol station, one fuel type, and one observation timestamp.

This dataset will be used to generate target labels and train the ML model.


In [2]:

# Map Ministry fuel names to SmartFuel fuel types
fuel_columns = {
    "Precio Gasolina 95 E5": "gasolina_95_e5",
    "Precio Gasoleo A": "gasoleo_a"
}

records = []

# Read all historical snapshots
for file in json_files:

    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    timestamp = pd.to_datetime(data["Fecha"], dayfirst=True)

    for station in data["ListaEESSPrecio"]:

        station_id = str(station.get("IDEESS", "")).strip()

        if not station_id:
            continue

        for column, fuel_type in fuel_columns.items():

            raw_price = str(station.get(column, "")).strip()

            if not raw_price:
                continue

            # Convert Spanish decimal commas into decimal points
            try:
                price = float(raw_price.replace(",", "."))
            except ValueError:
                continue

            if price <= 0:
                continue

            records.append({
                "station_id": station_id,
                "fuel_type": fuel_type,
                "ts": timestamp,
                "price_eur": price,
                "province": station.get("Provincia", "")
            })

# Combine observations into one DataFrame
historical_prices_df = pd.DataFrame(records)

# Remove duplicated station/fuel/timestamp combinations
historical_prices_df = historical_prices_df.drop_duplicates(
    subset=["station_id", "fuel_type", "ts"]
)

# Sort observations chronologically
historical_prices_df = historical_prices_df.sort_values(
    ["station_id", "fuel_type", "ts"]
).reset_index(drop=True)

print("Total price observations:", len(historical_prices_df))
print("Unique petrol stations:",
      historical_prices_df["station_id"].nunique())
print("Unique timestamps:",
      historical_prices_df["ts"].nunique())
print("First timestamp:", historical_prices_df["ts"].min())
print("Last timestamp:", historical_prices_df["ts"].max())

print("\nObservations by fuel type:")
print(historical_prices_df["fuel_type"].value_counts())

display(historical_prices_df.head(10))


Total price observations: 511065
Unique petrol stations: 11365
Unique timestamps: 23
First timestamp: 2026-11-12 11:27:41
Last timestamp: 2026-11-17 13:00:08

Observations by fuel type:
fuel_type
gasoleo_a         259494
gasolina_95_e5    251571
Name: count, dtype: int64


,station_id,fuel_type,ts,price_eur,province
0,1,gasoleo_a,2026-11-12 11:27:41,1.799,BIZKAIA
1,1,gasoleo_a,2026-11-12 11:38:44,1.799,BIZKAIA
2,1,gasoleo_a,2026-11-12 11:39:06,1.799,BIZKAIA
3,1,gasoleo_a,2026-11-12 13:00:08,1.799,BIZKAIA
4,1,gasoleo_a,2026-11-12 19:00:08,1.799,BIZKAIA
5,1,gasoleo_a,2026-11-13 01:00:08,1.799,BIZKAIA
6,1,gasoleo_a,2026-11-13 07:00:08,1.799,BIZKAIA
7,1,gasoleo_a,2026-11-13 13:00:08,1.799,BIZKAIA
8,1,gasoleo_a,2026-11-13 19:00:08,1.789,BIZKAIA
9,1,gasoleo_a,2026-11-14 01:00:08,1.789,BIZKAIA



## 8. Target Variable Creation

Create the target variable for supervised learning by comparing each fuel price with its next observed price at the same petrol station.

The target variable has three categories:

- `up`: Fuel price increases.
- `down`: Fuel price decreases.
- `flat`: Fuel price remains unchanged.

We exclude observations without a future price and unusually long gaps between snapshots.

This prepares the historical dataset for training and evaluating the ML classification model.


In [3]:

import numpy as np

# Start from the complete historical dataset
ml_df = historical_prices_df.copy()

# Sort observations chronologically
ml_df = ml_df.sort_values(
    ["station_id", "fuel_type", "ts"]
)

# Group prices by station and fuel type
group = ml_df.groupby(["station_id", "fuel_type"])

# Get the next observed price and timestamp
ml_df["next_price_eur"] = group["price_eur"].shift(-1)
ml_df["next_ts"] = group["ts"].shift(-1)

# Remove observations without a future price
ml_df = ml_df.dropna(
    subset=["next_price_eur", "next_ts"]
).copy()

# Calculate hours between observations
ml_df["hours_to_next"] = (
    (ml_df["next_ts"] - ml_df["ts"])
    .dt.total_seconds() / 3600
)

# Exclude missing observations and gaps longer than 8 hours
# Normal scheduled snapshots are approximately 6 hours apart
ml_df = ml_df[
    (ml_df["hours_to_next"] > 0) &
    (ml_df["hours_to_next"] <= 8)
].copy()

# Calculate price variation
ml_df["price_change_eur"] = (
    ml_df["next_price_eur"] - ml_df["price_eur"]
).round(3)

# Create classification target
ml_df["direction"] = np.select(
    [
        ml_df["price_change_eur"] > 0,
        ml_df["price_change_eur"] < 0
    ],
    ["up", "down"],
    default="flat"
)

# Check target distribution
print("Total labeled observations:", len(ml_df))

print("\nTarget distribution:")
print(ml_df["direction"].value_counts())

print("\nTarget distribution (%):")
print(
    ml_df["direction"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

display(ml_df.head(10))


Total labeled observations: 466512

Target distribution:
direction
flat    409935
down     43053
up       13524
Name: count, dtype: int64

Target distribution (%):
direction
flat    87.87
down     9.23
up       2.90
Name: proportion, dtype: float64


,station_id,fuel_type,ts,price_eur,province,next_price_eur,next_ts,hours_to_next,price_change_eur,direction
0,1,gasoleo_a,2026-11-12 11:27:41,1.799,BIZKAIA,1.799,2026-11-12 11:38:44,0.184167,0.00,flat
1,1,gasoleo_a,2026-11-12 11:38:44,1.799,BIZKAIA,1.799,2026-11-12 11:39:06,0.006111,0.00,flat
2,1,gasoleo_a,2026-11-12 11:39:06,1.799,BIZKAIA,1.799,2026-11-12 13:00:08,1.350556,0.00,flat
3,1,gasoleo_a,2026-11-12 13:00:08,1.799,BIZKAIA,1.799,2026-11-12 19:00:08,6.000000,0.00,flat
4,1,gasoleo_a,2026-11-12 19:00:08,1.799,BIZKAIA,1.799,2026-11-13 01:00:08,6.000000,0.00,flat
5,1,gasoleo_a,2026-11-13 01:00:08,1.799,BIZKAIA,1.799,2026-11-13 07:00:08,6.000000,0.00,flat
6,1,gasoleo_a,2026-11-13 07:00:08,1.799,BIZKAIA,1.799,2026-11-13 13:00:08,6.000000,0.00,flat
7,1,gasoleo_a,2026-11-13 13:00:08,1.799,BIZKAIA,1.789,2026-11-13 19:00:08,6.000000,-0.01,down
8,1,gasoleo_a,2026-11-13 19:00:08,1.789,BIZKAIA,1.789,2026-11-14 01:00:08,6.000000,0.00,flat
9,1,gasoleo_a,2026-11-14 01:00:08,1.789,BIZKAIA,1.789,2026-11-14 07:00:08,6.000000,0.00,flat



## 9. Train/Test Split and Baseline Evaluation

Split the historical dataset chronologically to avoid data leakage.

- Training period: November 12–15, 2026.
- Testing period: November 16–17, 2026.

Training labels must be known before the test period begins.

Evaluate a simple baseline that always predicts `flat`.

Because fuel prices rarely change, accuracy alone is not sufficient. We will also evaluate performance for the `up` and `down` classes.


In [4]:

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    balanced_accuracy_score
)

# Chronological split
split_date = pd.Timestamp("2026-11-16 00:00:00")

# Training: both current and next prices are before the split
train_df = ml_df[
    ml_df["next_ts"] < split_date
].copy()

# Testing: current observations begin after the split
test_df = ml_df[
    ml_df["ts"] >= split_date
].copy()

print("Training observations:", len(train_df))
print("Testing observations:", len(test_df))

print("\nTraining target distribution (%):")
print(
    train_df["direction"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution (%):")
print(
    test_df["direction"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# Baseline: always predict no price change
y_test = test_df["direction"]

baseline_predictions = np.full(
    len(test_df),
    "flat"
)

print("\nBaseline Accuracy:",
      round(accuracy_score(y_test, baseline_predictions), 4))

print("Baseline Balanced Accuracy:",
      round(balanced_accuracy_score(
          y_test, baseline_predictions
      ), 4))

print("\nBaseline Classification Report:")
print(
    classification_report(
        y_test,
        baseline_predictions,
        labels=["down", "flat", "up"],
        zero_division=0
    )
)


Training observations: 310835
Testing observations: 133467

Training target distribution (%):
direction
flat    88.27
down     9.07
up       2.67
Name: proportion, dtype: float64

Testing target distribution (%):
direction
flat    86.06
down    10.10
up       3.84
Name: proportion, dtype: float64

Baseline Accuracy: 0.8606
Baseline Balanced Accuracy: 0.3333

Baseline Classification Report:
              precision    recall  f1-score   support

        down       0.00      0.00      0.00     13481
        flat       0.86      1.00      0.93    114857
          up       0.00      0.00      0.00      5129

    accuracy                           0.86    133467
   macro avg       0.29      0.33      0.31    133467
weighted avg       0.74      0.86      0.80    133467




## 10. Train Machine Learning Model v0

Train a Random Forest Classifier to predict fuel-price direction (`up`, `down`, `flat`).

Use features available at prediction time:
- Current fuel price
- Day of the week
- Hour of the day
- Average price in the province
- Difference from the provincial average

Use class balancing to address the high proportion of unchanged prices.

Evaluate the model on the chronological test set and compare it against the `flat` baseline.


In [6]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


# 1. Create features using the complete current snapshots
# This matches how the AWS prediction Lambda calculates features.

feature_source = historical_prices_df.copy()

feature_source["weekday"] = feature_source["ts"].dt.weekday
feature_source["hour"] = feature_source["ts"].dt.hour

# Provincial mean using ALL valid prices in each snapshot
feature_source["province_mean"] = (
    feature_source.groupby(
        ["ts", "province", "fuel_type"],
        dropna=False
    )["price_eur"].transform("mean")
)

feature_source["price_vs_province"] = (
    feature_source["price_eur"] -
    feature_source["province_mean"]
)

# Attach features to the labeled dataset
extra_features = [
    "weekday",
    "hour",
    "province_mean",
    "price_vs_province"
]

ml_df = ml_df.drop(
    columns=extra_features,
    errors="ignore"
)

feature_lookup = feature_source.set_index(
    ["station_id", "fuel_type", "ts"]
)[extra_features]

ml_df = ml_df.join(
    feature_lookup,
    on=["station_id", "fuel_type", "ts"]
)


# 2. Select features compatible with SmartFuel's predict Lambda
features = [
    "price_eur",
    "weekday",
    "hour",
    "province_mean",
    "price_vs_province"
]

# 3. Use the chronological split from Section 9
X_train = ml_df.loc[train_df.index, features]
X_test = ml_df.loc[test_df.index, features]

y_train = train_df["direction"]
y_test = test_df["direction"]

# Check for missing feature values
assert X_train.notna().all().all()
assert X_test.notna().all().all()

# 4. Train Random Forest model
model = RandomForestClassifier(
    n_estimators=80,
    max_depth=12,
    min_samples_leaf=30,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 5. Predict the test set
model_predictions = model.predict(X_test)

# 6. Compare model against baseline
results = pd.DataFrame({
    "Model": ["Baseline (always flat)", "Random Forest v0"],
    "Accuracy": [
        accuracy_score(y_test, baseline_predictions),
        accuracy_score(y_test, model_predictions)
    ],
    "Balanced Accuracy": [
        balanced_accuracy_score(y_test, baseline_predictions),
        balanced_accuracy_score(y_test, model_predictions)
    ],
    "Macro F1": [
        f1_score(y_test, baseline_predictions, average="macro"),
        f1_score(y_test, model_predictions, average="macro")
    ]
})

print("Model comparison:")
display(results.round(4))

print("\nRandom Forest Classification Report:")
print(classification_report(
    y_test,
    model_predictions,
    labels=["down", "flat", "up"],
    zero_division=0
))

print("\nConfusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(
        y_test,
        model_predictions,
        labels=["down", "flat", "up"]
    ),
    index=["Actual down", "Actual flat", "Actual up"],
    columns=["Predicted down", "Predicted flat", "Predicted up"]
))


Model comparison:


,Model,Accuracy,Balanced Accuracy,Macro F1
0,Baseline (always flat),0.8606,0.3333,0.3084
1,Random Forest v0,0.7333,0.4518,0.4199



Random Forest Classification Report:
              precision    recall  f1-score   support

        down       0.26      0.40      0.31     13481
        flat       0.90      0.80      0.85    114857
          up       0.07      0.15      0.10      5129

    accuracy                           0.73    133467
   macro avg       0.41      0.45      0.42    133467
weighted avg       0.81      0.73      0.77    133467


Confusion Matrix:


,Predicted down,Predicted flat,Predicted up
Actual down,5452,6601,1428
Actual flat,14505,91638,8714
Actual up,1258,3086,785



## 11. Model Evaluation and Feature Importance

Analyze which features contribute most to the Random Forest predictions.

Evaluate the model's ability to identify fuel-price increases and decreases, considering the strong class imbalance.

The objective is to understand model behavior and identify potential improvements before deployment.


In [7]:

# Calculate feature importance
feature_importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print("Feature Importance:")
display(feature_importance.round(4))

# Compare actual and predicted class distributions
actual_distribution = (
    y_test.value_counts(normalize=True)
    .reindex(["down", "flat", "up"], fill_value=0)
    .mul(100)
)

predicted_distribution = (
    pd.Series(model_predictions)
    .value_counts(normalize=True)
    .reindex(["down", "flat", "up"], fill_value=0)
    .mul(100)
)

distribution_comparison = pd.DataFrame({
    "Actual (%)": actual_distribution,
    "Predicted (%)": predicted_distribution
})

print("\nActual vs Predicted Distribution:")
display(distribution_comparison.round(2))


Feature Importance:


,Feature,Importance
0,hour,0.3356
1,price_vs_province,0.2699
2,price_eur,0.1577
3,province_mean,0.1367
4,weekday,0.1002



Actual vs Predicted Distribution:


,Actual (%),Predicted (%)
down,10.10,15.90
flat,86.06,75.92
up,3.84,8.19



## 12. Model Stability Evaluation

Evaluate the Random Forest separately for each test date.

Compare accuracy and Macro F1 against the baseline to identify potential performance differences between days.

This helps assess whether the model's performance is consistent over time before exporting it.


In [8]:

# Create daily evaluation dataset
daily_df = test_df[["ts", "direction"]].copy()

daily_df["predicted"] = model_predictions
daily_df["baseline"] = "flat"
daily_df["date"] = daily_df["ts"].dt.date

# Evaluate each test day separately
daily_results = []

for date, group in daily_df.groupby("date"):

    actual = group["direction"]
    predicted = group["predicted"]
    baseline = group["baseline"]

    daily_results.append({
        "Date": str(date),
        "Observations": len(group),
        "Baseline Accuracy": accuracy_score(actual, baseline),
        "Model Accuracy": accuracy_score(actual, predicted),
        "Baseline Macro F1": f1_score(
            actual, baseline,
            labels=["down", "flat", "up"],
            average="macro",
            zero_division=0
        ),
        "Model Macro F1": f1_score(
            actual, predicted,
            labels=["down", "flat", "up"],
            average="macro",
            zero_division=0
        )
    })

daily_results_df = pd.DataFrame(daily_results)

print("Daily Model Performance:")
display(daily_results_df.round(4))


Daily Model Performance:


,Date,Observations,Baseline Accuracy,Model Accuracy,Baseline Macro F1,Model Macro F1
0,2026-11-16,88935,0.8388,0.6487,0.3041,0.4108
1,2026-11-17,44532,0.9041,0.9023,0.3165,0.3166



## 13. Export Model v0

Save the trained Random Forest classifier as a candidate model.

Export:
- `model.joblib`: Trained classification model.
- `metadata.json`: Model version, features and evaluation metrics.

The model will remain a candidate until its compatibility with the prediction Lambda is verified.

Production deployment is not yet approved due to limited historical data and unstable daily performance.


In [9]:

import joblib
import json
from pathlib import Path
from datetime import datetime, timezone

# Save model artifacts locally
output_dir = Path("ml/models/candidates/v0")

# Handle execution from the ml/ directory
if not Path("ml").exists():
    output_dir = Path("models/candidates/v0")

output_dir.mkdir(parents=True, exist_ok=True)

# Save trained classifier
joblib.dump(model, output_dir / "model.joblib")

# Prepare metadata using actual evaluation results
metadata = {
    "version": "v0-random-forest",
    "model_type": "RandomForestClassifier",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "features": features,
    "classes": list(model.classes_),
    "training_observations": len(train_df),
    "testing_observations": len(test_df),
    "accuracy": round(
        accuracy_score(y_test, model_predictions), 4
    ),
    "balanced_accuracy": round(
        balanced_accuracy_score(y_test, model_predictions), 4
    ),
    "macro_f1": round(
        f1_score(
            y_test, model_predictions,
            average="macro"
        ), 4
    ),
    "baseline_accuracy": round(
        accuracy_score(y_test, baseline_predictions), 4
    ),
    "status": "candidate",
    "production_approved": False
}

with open(output_dir / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Model exported successfully!")
print("Folder:", output_dir)
print("Files:")
for file in output_dir.iterdir():
    print("-", file.name)

print("\nMetadata:")
print(json.dumps(metadata, indent=2))


Model exported successfully!
Folder: models/candidates/v0
Files:
- metadata.json
- model.joblib

Metadata:
{
  "version": "v0-random-forest",
  "model_type": "RandomForestClassifier",
  "created_at": "2026-11-17T16:33:36.722333+00:00",
  "features": [
    "price_eur",
    "weekday",
    "hour",
    "province_mean",
    "price_vs_province"
  ],
  "classes": [
    "down",
    "flat",
    "up"
  ],
  "training_observations": 310835,
  "testing_observations": 133467,
  "accuracy": 0.7333,
  "balanced_accuracy": 0.4518,
  "macro_f1": 0.4199,
  "baseline_accuracy": 0.8606,
  "status": "candidate",
  "production_approved": false
}



## 14. Validate Exported Model

Reload the exported model and metadata to verify that the saved artifacts work correctly.

Check that:
- The model can be loaded using joblib.
- The feature names match the metadata.
- The model predicts valid directions (up, down, flat).
- Prediction probabilities are between 0 and 1 and sum to 1.

This validation is required before integrating the model with the AWS prediction Lambda.


In [10]:

# Reload saved model and metadata
loaded_model = joblib.load(output_dir / "model.joblib")

with open(output_dir / "metadata.json", "r") as f:
    loaded_metadata = json.load(f)

# Verify feature compatibility
assert list(loaded_model.feature_names_in_) == loaded_metadata["features"]

# Select 5 observations for testing
sample = X_test.head(5)

# Generate predictions
predictions = loaded_model.predict(sample)
probabilities = loaded_model.predict_proba(sample)

# Verify output
assert set(predictions).issubset({"up", "down", "flat"})
assert np.all((probabilities >= 0) & (probabilities <= 1))
assert np.allclose(probabilities.sum(axis=1), 1)

print("Model loaded successfully!")
print("Model version:", loaded_metadata["version"])
print("Features:", loaded_metadata["features"])
print("Classes:", list(loaded_model.classes_))

print("\nSample predictions:")

prediction_results = pd.DataFrame({
    "Predicted direction": predictions,
    "Confidence": probabilities.max(axis=1).round(4)
})

display(prediction_results)


Model loaded successfully!
Model version: v0-random-forest
Features: ['price_eur', 'weekday', 'hour', 'province_mean', 'price_vs_province']
Classes: ['down', 'flat', 'up']

Sample predictions:


,Predicted direction,Confidence
0,flat,0.7144
1,flat,0.6640
2,down,0.5302
3,flat,0.6170
4,flat,0.8539



## 15. Integration Test with SmartFuel Predict Lambda

Verify that the exported Random Forest v0 works with the existing Phase B prediction Lambda.

The test will:
- Load the actual Lambda prediction code.
- Build features from a historical fuel-price snapshot.
- Load the local model artifacts through a simulated S3 client.
- Generate predictions using the Lambda's prediction interface.

No AWS resources will be modified during this test.


In [11]:

import importlib.util
import io
from pathlib import Path

# 1. Locate Carla's predict Lambda
root = Path.cwd()

if not (root / "src/lambdas/predict/handler.py").exists():
    root = root.parent

handler_path = root / "src/lambdas/predict/handler.py"

spec = importlib.util.spec_from_file_location(
    "smartfuel_predict_handler",
    handler_path
)

predict_handler = importlib.util.module_from_spec(spec)
spec.loader.exec_module(predict_handler)

# 2. Simulate S3 using our exported local files
class LocalS3:
    def get_object(self, Bucket, Key):
        filename = Key.split("/")[-1]
        file_path = output_dir / filename

        return {
            "Body": io.BytesIO(file_path.read_bytes())
        }

predict_handler._s3 = LocalS3()

# 3. Select the most recent historical snapshot
latest_ts = historical_prices_df["ts"].max()

snapshot_df = historical_prices_df[
    historical_prices_df["ts"] == latest_ts
].copy()

# Required by Carla's feature-building function.
# Placeholder for this local test only; our model
# does not use price_rank_in_geohash.
snapshot_df["geohash5"] = "local-test"

# 4. Build features using Carla's Lambda
lambda_features = predict_handler.build_features(
    snapshot_df
)

# 5. Load our model through Carla's load_model()
predict_fn, version = predict_handler.load_model(
    "local-test-bucket",
    "models/candidates/v0/"
)

# 6. Generate predictions
sample_features = lambda_features.head(5)

predictions = predict_fn(sample_features)

# 7. Validate compatibility
assert version == loaded_metadata["version"]
assert len(predictions) == 5

assert all(
    p["direction"] in ["up", "down", "flat"]
    and 0 <= p["prob"] <= 1
    for p in predictions
)

print("Integration test successful!")
print("Model version:", version)
print("Predictions generated:", len(predictions))

display(pd.DataFrame(predictions))


Integration test successful!
Model version: v0-random-forest
Predictions generated: 5


,direction,prob
0,down,0.582696
1,down,0.629203
2,up,0.472018
3,up,0.465769
4,up,0.514343
